# 🌸 Iris Flower Classification
### Datheon – Machine Learning and Data Science Internship | Task 1

This notebook builds and compares two supervised multiclass classification models using the Iris dataset:
- Logistic Regression
- Decision Tree Classifier

**Workflow:** data loading → inspection → EDA → train/test split → preprocessing → model training → evaluation → visualization.

> Dataset source: `sklearn.datasets.load_iris` (150 samples, 4 numeric features, 3 classes).

## 1. Import libraries
Load the Python libraries used for data handling, visualization and machine learning.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

# When this notebook is run from the Notebook/ folder, outputs are saved
# in the sibling images/ folder at the project root.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebook" else Path.cwd()
IMAGE_DIR = PROJECT_DIR / "images"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
print("Libraries imported successfully.")
print("Images will be saved to:", IMAGE_DIR.resolve())

## 2. Load the Iris dataset
The dataset is loaded directly from scikit-learn, so no external CSV file is required.

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df = df.rename(columns={"target": "species_id"})
df["species"] = df["species_id"].map(dict(enumerate(iris.target_names)))

feature_names = list(iris.feature_names)
target_names = list(iris.target_names)

display(df.head())
print(f"Dataset shape: {df.shape}")

## 3. Data inspection
Review data types, missing values, class counts and descriptive statistics.

In [ ]:
print("Dataset information:")
df.info()

print("\\nMissing values:")
display(df.isna().sum().to_frame("missing_values"))

print("\\nClass distribution:")
display(df["species"].value_counts().rename_axis("species").to_frame("count"))

print("\\nDescriptive statistics:")
display(df[feature_names].describe().T)

## 4. Exploratory Data Analysis (EDA)
The following plots are saved as PNG files in the project-level `images/` folder.

In [ ]:
# 4.1 Species distribution
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x="species", order=target_names)
ax.set(title="Distribution of Iris Flower Species",
       xlabel="Species", ylabel="Number of samples")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "species_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 4.2 Pairplot
pair_grid = sns.pairplot(
    df, vars=feature_names, hue="species",
    hue_order=target_names, diag_kind="hist"
)
pair_grid.fig.suptitle("Pairplot of Iris Dataset", y=1.02)
pair_grid.savefig(IMAGE_DIR / "pairplot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 4.3 Correlation heatmap (numeric features only)
plt.figure(figsize=(9, 7))
corr = df[feature_names].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Correlation Heatmap of Iris Features")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Prepare features and target
Use a stratified train/test split to preserve the class proportions in both sets.

In [ ]:
X = df[feature_names]
y = df["species_id"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features: ", X_test.shape)
print("\\nTraining class distribution:")
display(y_train.value_counts().sort_index().rename(index=dict(enumerate(target_names))))

## 6. Build and train classification models

**Logistic Regression:** StandardScaler is included in a scikit-learn Pipeline. The scaler is fitted only on training data, preventing data leakage.

**Decision Tree:** A tree-based model trained on the original feature values. Feature scaling is not required for decision trees.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=500, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE)
}

predictions = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions[model_name] = model.predict(X_test)
    print(f"{model_name} trained successfully.")

## 7. Evaluate model performance
Compare accuracy, precision, recall and F1-score. Macro averages give equal weight to each class.

In [ ]:
summary_rows = []
for model_name, y_pred in predictions.items():
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    summary_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1-score": f1
    })

comparison = pd.DataFrame(summary_rows).set_index("Model")
display(comparison.round(4))

In [ ]:
for model_name, y_pred in predictions.items():
    print(f"\\n{model_name} – Classification Report")
    print(classification_report(
        y_test, y_pred, target_names=target_names, zero_division=0
    ))

## 8. Confusion matrices
Each matrix compares actual labels (rows) with predicted labels (columns).

In [ ]:
confusion_image_names = {
    "Logistic Regression": "logistic_regression_confusion_matrix.png",
    "Decision Tree": "decision_tree_confusion_matrix.png"
}

for model_name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=target_names, yticklabels=target_names
    )
    plt.title(f"{model_name} – Confusion Matrix")
    plt.xlabel("Predicted species")
    plt.ylabel("Actual species")
    plt.tight_layout()
    plt.savefig(IMAGE_DIR / confusion_image_names[model_name],
                dpi=300, bbox_inches="tight")
    plt.show()

## 9. Decision boundary visualization

For a 2D illustration, the models below use **petal length** and **petal width** only. These visualizations are illustrative and are not the same as the four-feature models evaluated above.

In [ ]:
boundary_features = ["petal length (cm)", "petal width (cm)"]
X_2d = df[boundary_features]
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y, test_size=TEST_SIZE,
    random_state=RANDOM_STATE, stratify=y
)

boundary_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=500, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE)
}

boundary_image_names = {
    "Logistic Regression": "logistic_regression_boundary.png",
    "Decision Tree": "decision_tree_boundary.png"
}

for model_name, model in boundary_models.items():
    model.fit(X_train_2d, y_train_2d)

    # Generate a grid in original feature units. The pipeline scales internally
    # for Logistic Regression; the Decision Tree uses the raw values.
    x_min, x_max = X_2d.iloc[:, 0].min() - 0.5, X_2d.iloc[:, 0].max() + 0.5
    y_min, y_max = X_2d.iloc[:, 1].min() - 0.5, X_2d.iloc[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250)
    )
    grid = pd.DataFrame(
        np.c_[xx.ravel(), yy.ravel()],
        columns=boundary_features
    )
    zz = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(9, 6))
    plt.contourf(xx, yy, zz, alpha=0.28, cmap="viridis")
    scatter = plt.scatter(
        X_train_2d.iloc[:, 0], X_train_2d.iloc[:, 1],
        c=y_train_2d, cmap="viridis", edgecolors="black", s=45
    )
    plt.title(f"{model_name} – Decision Boundary (2 Features)")
    plt.xlabel("Petal length (cm)")
    plt.ylabel("Petal width (cm)")
    plt.tight_layout()
    plt.savefig(IMAGE_DIR / boundary_image_names[model_name],
                dpi=300, bbox_inches="tight")
    plt.show()

## 10. Final model comparison
The best score depends on the selected evaluation metric. Both models are evaluated on the same held-out test set.

In [ ]:
display(comparison.sort_values("Accuracy", ascending=False).round(4))

best_model = comparison["Accuracy"].idxmax()
best_accuracy = comparison.loc[best_model, "Accuracy"]
print(f"Highest test accuracy: {best_model} ({best_accuracy:.2%})")

## 11. Conclusion

This notebook demonstrates a complete multiclass classification workflow using the Iris dataset. It includes data inspection, exploratory visualizations, a stratified train/test split, model training, evaluation metrics, confusion matrices and two-feature decision boundaries.

The evaluation results above are calculated when the notebook is run. Interpret model performance using the full set of metrics rather than accuracy alone.

## 12. Reproducibility

- Random state: `42`
- Test size: `20%`
- Dataset: Scikit-learn Iris dataset
- Generated figures: saved to `../images/` when run from this `Notebook/` directory.

**To run:** install the packages listed in `requirements.txt`, open this notebook in Jupyter, and select **Run All**.